<a href="https://colab.research.google.com/github/ggalarza1/accounting/blob/main/Bitcoin_wallet_data_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!apt-get update
!apt-get install -y libsecp256k1-dev
!pip install base58 bip32 ecdsa

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,030 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,934 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/restricte

In [7]:
#!/usr/bin/env python3
"""
Bitcoin Wallet Data Extractor with Historical Pricing + xpub Support
Uses free mempool.space API + CoinGecko for transaction history with cost basis
Supports single addresses OR full HD wallet scanning via xpub
No API key required - perfect for prototyping

Requires: pip install bip32 ecdsa (for xpub support)
"""

import requests
import json
import time
import hashlib
#import base58
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

# Try to import bip32 for xpub support
try:
    from bip32 import BIP32
    BIP32_AVAILABLE = True
except ImportError:
    BIP32_AVAILABLE = False
    print("🔔 Note: bip32 library not installed. xpub scanning will be limited.")
    print("   Install with: pip install bip32 ecdsa")

class XpubScanner:
    """
    Scan all addresses derived from an xpub (Extended Public Key)
    Implements BIP32/BIP44 derivation for HD wallets
    """

    def __init__(self, xpub: str, gap_limit: int = 20):
        """
        Initialize xpub scanner

        Args:
            xpub: Extended public key (xpub, ypub, zpub supported)
            gap_limit: Stop scanning after N consecutive empty addresses
        """
        self.xpub = xpub
        self.gap_limit = gap_limit
        self.session = requests.Session()

        if not BIP32_AVAILABLE:
            raise ImportError("bip32 library required. Install with: pip install bip32 ecdsa")

        self.bip32 = BIP32.from_xpub(xpub)

        # Determine derivation path based on xpub prefix
        self.is_segwit = self._detect_segwit_type()

    def _detect_segwit_type(self) -> str:
        """Detect address type (Legacy/Segwit/Taproot) from xpub variant"""
        if self.xpub.startswith('zpub') or self.xpub.startswith('vpub'):
            return 'bech32'  # Native Segwit
        elif self.xpub.startswith('ypub') or self.xpub.startswith('upub'):
            return 'p2sh-segwit'  # Nested Segwit
        else:
            return 'legacy'  # xpub

    def _get_address_from_pubkey(self, public_key: bytes, address_type: str) -> str:
        """Convert public key to Bitcoin address"""
        import hashlib

        # Hash public key (sha256 then ripemd160)
        sha256_hash = hashlib.sha256(public_key).digest()

        try:
            import hashlib
            ripemd160 = hashlib.new('ripemd160')
            ripemd160.update(sha256_hash)
            pub_key_hash = ripemd160.digest()
        except:
            # Fallback if ripemd160 not available
            return None

        if address_type == 'bech32':
            # Native Segwit (bc1... addresses)
            return self._hash_to_bech32(pub_key_hash)
        elif address_type == 'p2sh-segwit':
            # P2SH-wrapped Segwit (3... addresses)
            return self._hash_to_p2sh(pub_key_hash)
        else:
            # Legacy (1... addresses)
            return self._hash_to_p2pkh(pub_key_hash)

    def _hash_to_bech32(self, hash_bytes: bytes) -> str:
        """Convert to bech32 address (simplified - use libraries for production)"""
        # Note: Full bech32 implementation requires bech32 library
        # This is a placeholder - install bech32 library for full support
        return f"bc1q{hash_bytes[:20].hex()}... (requires bech32 library)"

    def _hash_to_p2sh(self, hash_bytes: bytes) -> str:
        """Convert to P2SH address"""
        # Placeholder - requires proper P2SH script building
        return f"3... (requires script building)"

    def _hash_to_p2pkh(self, hash_bytes: bytes) -> str:
        """Convert to legacy P2PKH address (1...)"""
        # Add version byte (0x00 for mainnet)
        versioned = b'\x00' + hash_bytes
        checksum = hashlib.sha256(hashlib.sha256(versioned).digest()).digest()[:4]
        address_bytes = versioned + checksum
        return base58.b58encode(address_bytes).decode('ascii')

    def derive_address(self, account: int, chain: int, index: int) -> Tuple[str, str]:
        """
        Derive a single address

        Args:
            account: Account number (usually 0 for default)
            chain: 0 = receiving (external), 1 = change (internal)
            index: Address index

        Returns:
            Tuple of (address, full_derivation_path)
        """
        # BIP44 derivation: m / 44' / coin_type' / account' / chain / index
        # For xpub, we start after the hardened derivation
        path = f"{account}/{chain}/{index}"

        try:
            public_key = self.bip32.get_pubkey_from_path(path)
            address = self._get_address_from_pubkey(public_key, self.is_segwit)
            return address, path
        except Exception as e:
            print(f"Error deriving address at {path}: {e}")
            return None, path

    def discover_active_addresses(self, account: int = 0, max_addresses: int = 1000) -> Dict[str, List[Dict]]:
        """
        Discover all addresses with transaction history

        Uses gap limit: stop scanning after N consecutive empty addresses

        Args:
            account: Account number
            max_addresses: Safety limit

        Returns:
            Dict with 'external' (receiving) and 'change' lists of used addresses
        """
        active_addresses = {
            'external': [],  # chain 0 - receiving
            'change': []     # chain 1 - internal change
        }

        base_url = "https://mempool.space/api"

        for chain_type, chain_index in [('external', 0), ('change', 1)]:
            print(f"\n🔍 Scanning {chain_type} addresses...")

            gap_count = 0
            index = 0

            while gap_count < self.gap_limit and index < max_addresses:
                address, path = self.derive_address(account, chain_index, index)

                if not address or 'requires' in address:
                    # Fallback for address generation - use the public key as identifier
                    print(f"  Note: Skipping address generation ({address}), use bech32 library for full support")
                    index += 1
                    continue

                # Check if address has transactions
                url = f"{base_url}/address/{address}/txs"
                try:
                    response = self.session.get(url, timeout=10)
                    if response.status_code == 429:
                        time.sleep(2)
                        continue

                    response.raise_for_status()
                    txs = response.json()

                    if len(txs) > 0:
                        print(f"  ✅ Index {index}: {len(txs)} transactions")
                        active_addresses[chain_type].append({
                            'address': address,
                            'index': index,
                            'path': path,
                            'transaction_count': len(txs),
                            'transactions': txs
                        })
                        gap_count = 0  # Reset gap
                    else:
                        gap_count += 1

                except Exception as e:
                    print(f"  Error checking address {index}: {e}")
                    gap_count += 1

                index += 1

        return active_addresses

    def get_all_addresses(self) -> List[str]:
        """Get all discovered addresses as a simple list"""
        discovered = self.discover_active_addresses()
        addresses = []
        for chain in ['external', 'change']:
            addresses.extend([a['address'] for a in discovered[chain]])
        return addresses


class BitcoinPriceLookup:
    """Fetch historical Bitcoin prices in USD using CoinGecko API (free tier)"""

    def __init__(self, base_url: str = "https://api.coingecko.com/api/v3"):
        self.base_url = base_url
        self.session = requests.Session()
        self._price_cache = {}  # Cache prices to minimize API calls
        self._last_request_time = datetime.now()
        self._rate_limit_delay = 1.2  # Seconds between requests (free tier: 10-30 calls/min)

    def _respect_rate_limit(self):
        """Ensure we don't hit CoinGecko rate limits"""
        elapsed = (datetime.now() - self._last_request_time).total_seconds()
        if elapsed < self._rate_limit_delay:
            time.sleep(self._rate_limit_delay - elapsed)
        self._last_request_time = datetime.now()

    def get_price_on_date(self, date_str: str) -> Optional[float]:
        """
        Fetch Bitcoin price in USD for a specific date

        Args:
            date_str: Date in format 'YYYY-MM-DD'

        Returns:
            Price in USD or None if unavailable
        """
        # Check cache first
        if date_str in self._price_cache:
            return self._price_cache[date_str]

        self._respect_rate_limit()

        url = f"{self.base_url}/coins/bitcoin/history"
        params = {
            'date': datetime.strptime(date_str, '%Y-%m-%d').strftime('%d-%m-%Y'),
            'localization': 'false'
        }

        try:
            response = self.session.get(url, params=params, timeout=30)

            if response.status_code == 429:
                print(f"Rate limit hit for {date_str}, waiting 5 seconds...")
                time.sleep(5)
                return self.get_price_on_date(date_str)

            response.raise_for_status()
            data = response.json()

            price = data.get('market_data', {}).get('current_price', {}).get('usd')
            if price:
                self._price_cache[date_str] = price
            return price

        except requests.exceptions.RequestException as e:
            print(f"Error fetching price for {date_str}: {e}")
            return None

    def get_price_for_timestamp(self, timestamp: int) -> Optional[float]:
        """Convert timestamp to date string and fetch price"""
        date_str = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d')
        return self.get_price_on_date(date_str)


class BitcoinWalletExtractor:
    def __init__(self, base_url: str = "https://mempool.space/api"):
        self.base_url = base_url
        self.session = requests.Session()
        self.price_lookup = BitcoinPriceLookup()
        self._is_xpub_mode = False
        self._scanned_addresses = []

    def is_xpub(self, key: str) -> bool:
        """Check if key is an extended public key (xpub, ypub, zpub, etc.)"""
        if not key:
            return False
        # Remove whitespace and check prefix
        key = key.strip()
        extended_prefixes = ['xpub', 'ypub', 'zpub', 'tpub', 'upub', 'vpub']
        return any(key.startswith(p) for p in extended_prefixes)

    def scan_xpub(self, xpub: str, gap_limit: int = 20) -> Dict:
        """
        Scan all addresses from an xpub and aggregate transactions

        Args:
            xpub: Extended public key
            gap_limit: Consecutive empty addresses before stopping

        Returns:
            Aggregated wallet data across all discovered addresses
        """
        if not BIP32_AVAILABLE:
            print("❌ xpub scanning requires 'bip32' library")
            print("   Install with: pip install bip32 ecdsa")
            return {}

        print(f"\n⚡ Scanning xpub for active addresses (gap limit: {gap_limit})...")
        self._is_xpub_mode = True

        scanner = XpubScanner(xpub, gap_limit)
        discovered = scanner.discover_active_addresses(gap_limit=gap_limit)

        # Collect all addresses
        all_addresses = []
        all_transactions = {}
        address_info = {}

        for chain_type in ['external', 'change']:
            for addr_data in discovered[chain_type]:
                address = addr_data['address']
                all_addresses.append(address)
                address_info[address] = {
                    'path': addr_data['path'],
                    'index': addr_data['index'],
                    'type': chain_type
                }

                # Get detailed transactions for this address
                print(f"Fetching transactions for {address[:20]}...")
                txs = self.get_address_transactions(address)
                all_transactions[address] = txs

        self._scanned_addresses = all_addresses

        # Aggregate all transactions
        aggregated = self._aggregate_multi_address_data(all_addresses, all_transactions)
        aggregated['xpub'] = xpub
        aggregated['discovered_addresses'] = all_addresses
        aggregated['address_count'] = len(all_addresses)
        aggregated['address_info'] = address_info

        return aggregated

    def _aggregate_multi_address_data(self, addresses: List[str],
                                       address_transactions: Dict) -> Dict:
        """
        Aggregate transaction data from multiple addresses
        Apply FIFO cost basis across ALL transactions (not per-address)
        """
        all_parsed_txs = []

        for address in addresses:
            txs = address_transactions.get(address, [])
            for tx in txs:
                parsed = self.parse_transaction(tx, address)
                parsed['source_address'] = address  # Track which address
                all_parsed_txs.append(parsed)

        # Sort globally by timestamp for proper FIFO
        all_parsed_txs.sort(key=lambda x: x['date'] if x['date'] != 'pending' else '9999-12-31')

        # Aggregate totals
        total_received_btc = sum(tx['btc_received'] for tx in all_parsed_txs)
        total_sent_btc = sum(tx['btc_sent'] for tx in all_parsed_txs)
        total_fees_btc = sum(tx['btc_fee'] for tx in all_parsed_txs if tx['type'] == 'outgoing')

        total_usd_received = sum(tx['usd_received'] for tx in all_parsed_txs if tx['usd_received'])
        total_usd_sent = sum(tx['usd_sent'] for tx in all_parsed_txs if tx['usd_sent'])

        # Global FIFO cost basis across all addresses
        fifo_queue = []
        total_realized_gains = 0

        for tx in all_parsed_txs:
            if tx['type'] == 'incoming' and tx['btc_received'] > 0 and tx['usd_price']:
                fifo_queue.append({
                    'btc': tx['btc_received'],
                    'cost_basis': tx['usd_price'],
                    'address': tx['source_address']
                })

            elif tx['type'] == 'outgoing' and tx['btc_sent'] > 0 and tx['usd_price']:
                btc_to_sell = tx['btc_sent']
                cost_basis_for_sale = 0

                while btc_to_sell > 0 and fifo_queue:
                    oldest = fifo_queue[0]
                    btc_from_oldest = min(btc_to_sell, oldest['btc'])

                    cost_basis_for_sale += btc_from_oldest * oldest['cost_basis']
                    oldest['btc'] -= btc_from_oldest
                    btc_to_sell -= btc_from_oldest

                    if oldest['btc'] <= 0:
                        fifo_queue.pop(0)

                sale_proceeds = tx['btc_sent'] * tx['usd_price']
                gain_loss = sale_proceeds - cost_basis_for_sale
                total_realized_gains += gain_loss
                tx['realized_gain_loss'] = gain_loss
                tx['cost_basis_used'] = cost_basis_for_sale

        # Current balance and unrealized gains
        current_balance = total_received_btc - total_sent_btc
        current_price = self.price_lookup.get_price_on_date(
            datetime.now().strftime('%Y-%m-%d')
        )

        unrealized_gains = 0
        if current_price and fifo_queue:
            current_value = current_balance * current_price
            remaining_cost_basis = sum(h['btc'] * h['cost_basis'] for h in fifo_queue)
            unrealized_gains = current_value - remaining_cost_basis

        return {
            'addresses': addresses,
            'transaction_count': len(all_parsed_txs),
            'current_balance_btc': current_balance,
            'total_received_btc': total_received_btc,
            'total_sent_btc': total_sent_btc,
            'total_fees_btc': total_fees_btc,
            'total_received_usd': total_usd_received,
            'total_sent_usd': total_usd_sent,
            'realized_gains_losses_usd': total_realized_gains,
            'unrealized_gains_losses_usd': unrealized_gains,
            'current_btc_price_usd': current_price,
            'transactions': all_parsed_txs
        }

    def get_address_transactions(self, address: str) -> List[Dict]:
        """
        Fetch all transactions for a given Bitcoin address

        Args:
            address: Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)

        Returns:
            List of transaction dictionaries
        """
        url = f"{self.base_url}/address/{address}/txs"

        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching transactions: {e}")
            return []

    def get_address_utxos(self, address: str) -> List[Dict]:
        """
        Fetch unspent transaction outputs (UTXOs) for an address
        These represent the current balance
        """
        url = f"{self.base_url}/address/{address}/utxo"

        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching UTXOs: {e}")
            return []

    def parse_transaction(self, tx: Dict, target_address: str) -> Dict:
        """
        Parse a raw transaction into accounting-friendly format with USD pricing

        Args:
            tx: Raw transaction dict from API
            target_address: The address we're analyzing

        Returns:
            Cleaned transaction record with accounting fields and USD values
        """
        # Convert timestamp to readable date
        timestamp = tx.get('status', {}).get('block_time')
        date_str = datetime.fromtimestamp(timestamp).isoformat() if timestamp else 'pending'
        date_only = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d') if timestamp else None

        # Calculate amounts
        inputs = tx.get('vin', [])
        outputs = tx.get('vout', [])

        # Check if our address is in inputs (sending) or outputs (receiving)
        is_sender = any(inp.get('prevout', {}).get('scriptpubkey_address') == target_address
                        for inp in inputs)
        is_receiver = any(out.get('scriptpubkey_address') == target_address
                         for out in outputs)

        # Calculate net amount for this address
        btc_received = 0
        btc_sent = 0

        if is_receiver:
            for out in outputs:
                if out.get('scriptpubkey_address') == target_address:
                    btc_received += out.get('value', 0)

        if is_sender:
            for inp in inputs:
                if inp.get('prevout', {}).get('scriptpubkey_address') == target_address:
                    btc_sent += inp.get('prevout', {}).get('value', 0)

        # Transaction fee (paid by sender)
        fee = tx.get('fee', 0)

        # Determine transaction type for accounting
        if is_sender and is_receiver:
            tx_type = "self_transfer"  # Sending to yourself (e.g., consolidation)
        elif is_sender:
            tx_type = "outgoing"
        elif is_receiver:
            tx_type = "incoming"
        else:
            tx_type = "unknown"

        # Fetch historical USD price for cost basis
        usd_price = None
        usd_received = None
        usd_sent = None
        usd_fee = None

        if timestamp and date_only:
            usd_price = self.price_lookup.get_price_on_date(date_only)
            if usd_price:
                usd_received = (btc_received / 1e8) * usd_price
                usd_sent = (btc_sent / 1e8) * usd_price
                usd_fee = (fee / 1e8) * usd_price if is_sender else 0

        return {
            'tx_id': tx.get('txid'),
            'date': date_str,
            'date_only': date_only,
            'block_height': tx.get('status', {}).get('block_height'),
            'confirmations': tx.get('status', {}).get('confirmed', False),
            'type': tx_type,
            'btc_received': btc_received / 1e8,
            'btc_sent': btc_sent / 1e8,
            'btc_fee': fee / 1e8 if is_sender else 0,
            'net_change_btc': (btc_received - btc_sent) / 1e8,
            'usd_price': usd_price,
            'usd_received': usd_received,
            'usd_sent': usd_sent,
            'usd_fee': usd_fee,
            'net_change_usd': (usd_received - usd_sent) if usd_received and usd_sent else None,
            'address': target_address
        }

    def get_wallet_summary(self, address: str) -> Dict:
        """
        Get a complete summary of wallet activity with cost basis and gains/losses
        """
        print(f"Fetching data for address: {address}")

        # Get all transactions
        transactions = self.get_address_transactions(address)

        if not transactions:
            print("No transactions found or error occurred")
            return {}

        print(f"Found {len(transactions)} transactions")
        print("Fetching historical prices for cost basis calculations...")

        # Parse each transaction (this will also fetch prices)
        parsed_txs = [self.parse_transaction(tx, address) for tx in transactions]

        # Sort by date for FIFO cost basis calculation
        parsed_txs.sort(key=lambda x: x['date'] if x['date'] != 'pending' else '9999-12-31')

        # Get current UTXOs (unspent = current balance)
        utxos = self.get_address_utxos(address)
        current_balance_sats = sum(utxo.get('value', 0) for utxo in utxos)

        # Calculate BTC totals
        total_received = sum(tx['btc_received'] for tx in parsed_txs)
        total_sent = sum(tx['btc_sent'] for tx in parsed_txs)
        total_fees = sum(tx['btc_fee'] for tx in parsed_txs if tx['type'] == 'outgoing')

        # Calculate USD totals
        total_usd_received = sum(tx['usd_received'] for tx in parsed_txs if tx['usd_received'])
        total_usd_sent = sum(tx['usd_sent'] for tx in parsed_txs if tx['usd_sent'])
        total_usd_fees = sum(tx['usd_fee'] for tx in parsed_txs if tx['usd_fee'])

        # FIFO Cost Basis Calculation for Realized Gains/Losses
        fifo_queue = []  # (btc_amount, usd_cost_basis_per_btc)
        total_realized_gains = 0

        for tx in parsed_txs:
            if tx['type'] == 'incoming' and tx['btc_received'] > 0 and tx['usd_price']:
                # Add to inventory (FIFO queue)
                fifo_queue.append({
                    'btc': tx['btc_received'],
                    'cost_basis': tx['usd_price']
                })

            elif tx['type'] == 'outgoing' and tx['btc_sent'] > 0 and tx['usd_price']:
                # Calculate cost basis for sold BTC using FIFO
                btc_to_sell = tx['btc_sent']
                cost_basis_for_sale = 0

                while btc_to_sell > 0 and fifo_queue:
                    oldest = fifo_queue[0]
                    btc_from_oldest = min(btc_to_sell, oldest['btc'])

                    cost_basis_for_sale += btc_from_oldest * oldest['cost_basis']
                    oldest['btc'] -= btc_from_oldest
                    btc_to_sell -= btc_from_oldest

                    if oldest['btc'] <= 0:
                        fifo_queue.pop(0)

                # Calculate realized gain/loss
                sale_proceeds = tx['btc_sent'] * tx['usd_price']
                gain_loss = sale_proceeds - cost_basis_for_sale
                total_realized_gains += gain_loss
                tx['realized_gain_loss'] = gain_loss
                tx['cost_basis_used'] = cost_basis_for_sale

            else:
                tx['realized_gain_loss'] = None
                tx['cost_basis_used'] = None

        # Current unrealized gains (current balance at current price)
        current_btc_price = self.price_lookup.get_price_on_date(
            datetime.now().strftime('%Y-%m-%d')
        )
        unrealized_gains = 0
        if current_btc_price and fifo_queue:
            current_value = (current_balance_sats / 1e8) * current_btc_price
            remaining_cost_basis = sum(h['btc'] * h['cost_basis'] for h in fifo_queue)
            unrealized_gains = current_value - remaining_cost_basis

        return {
            'address': address,
            'transaction_count': len(transactions),
            'current_balance_btc': current_balance_sats / 1e8,
            'total_received_btc': total_received,
            'total_sent_btc': total_sent,
            'total_fees_btc': total_fees,
            'total_received_usd': total_usd_received,
            'total_sent_usd': total_usd_sent,
            'total_fees_usd': total_usd_fees,
            'realized_gains_losses_usd': total_realized_gains,
            'unrealized_gains_losses_usd': unrealized_gains,
            'current_btc_price_usd': current_btc_price,
            'transactions': parsed_txs,
            'utxos': utxos
        }


def print_summary(summary: Dict, is_xpub: bool = False):
    """Pretty print the wallet summary with accounting data"""
    if not summary:
        print("No data to display")
        return

    print("\n" + "=" * 70)
    if is_xpub:
        print("HD WALLET SUMMARY (xpub)")
        print(f"xpub: {summary.get('xpub', 'N/A')[:40]}...")
        print(f"Addresses Found: {summary.get('address_count', 0)}")
    else:
        print(f"WALLET SUMMARY: {summary.get('address', 'N/A')}")
    print("=" * 70)

    # BTC Section
    print("\n📊 BITCOIN BALANCES")
    print("-" * 40)
    print(f"Current Balance:    {summary['current_balance_btc']:.8f} BTC")
    print(f"Total Received:     {summary['total_received_btc']:.8f} BTC")
    print(f"Total Sent:         {summary['total_sent_btc']:.8f} BTC")
    print(f"Total Fees Paid:    {summary['total_fees_btc']:.8f} BTC")
    print(f"Transaction Count:  {summary['transaction_count']}")

    # USD Section
    print("\n💵 USD VALUATION")
    print("-" * 40)
    print(f"Total Received:     ${summary.get('total_received_usd', 0):,.2f}")
    print(f"Total Sent:         ${summary.get('total_sent_usd', 0):,.2f}")
    print(f"Total Fees:         ${summary.get('total_fees_usd', 0):,.2f}")
    if summary.get('current_btc_price_usd'):
        print(f"Current BTC Price:  ${summary['current_btc_price_usd']:,.2f}")

    # Gains/Losses Section
    print("\n📈 TAX REPORTING (FIFO METHOD)")
    print("-" * 40)
    realized = summary.get('realized_gains_losses_usd', 0)
    unrealized = summary.get('unrealized_gains_losses_usd', 0)
    print(f"Realized Gains/Losses:   ${realized:,.2f} {'📈' if realized >= 0 else '📉'}")
    print(f"Unrealized Gains/Losses: ${unrealized:,.2f} {'📈' if unrealized >= 0 else '📉'}")
    print("=" * 70)

    # Show discovered addresses if xpub mode
    if is_xpub and summary.get('discovered_addresses'):
        print("\n📍 DISCOVERED ADDRESSES")
        print("-" * 40)
        for i, addr in enumerate(summary['discovered_addresses'][:5]):
            addr_info = summary.get('address_info', {}).get(addr, {})
            addr_type = addr_info.get('type', 'unknown')
            index = addr_info.get('index', 'N/A')
            print(f"  {i+1}. {addr[:30]}... [{addr_type}:{index}]")
        if len(summary['discovered_addresses']) > 5:
            print(f"  ... and {len(summary['discovered_addresses']) - 5} more")

    print("\n--- RECENT TRANSACTIONS ---")
    for tx in summary['transactions'][:5]:  # Show last 5
        direction = "📥" if tx['type'] == 'incoming' else "📤" if tx['type'] == 'outgoing' else "🔄"
        usd_str = f"${tx['net_change_usd']:,.2f}" if tx.get('net_change_usd') else "N/A"
        addr_suffix = f" | Addr: {tx.get('source_address', '')[:8]}..." if is_xpub else ""
        print(f"{direction} {tx['date'][:10]} | {tx['type'].upper():12} | "
              f"BTC: {tx['net_change_btc']:+.8f} | USD: {usd_str}{addr_suffix}")
        if tx.get('realized_gain_loss') is not None:
            gain_str = f"${tx['realized_gain_loss']:,.2f}"
            print(f"    └─ Gain/Loss: {gain_str} {'📈' if tx['realized_gain_loss'] >= 0 else '📉'}")

    if len(summary['transactions']) > 5:
        print(f"... and {len(summary['transactions']) - 5} more transactions")


def export_to_csv(summary: Dict, filename: str = 'wallet_report.csv'):
    """Export transactions to CSV for accounting software import"""
    import csv

    # Extended fieldnames for xpub mode
    base_fields = [
        'tx_id', 'date', 'type', 'btc_received', 'btc_sent', 'btc_fee',
        'net_change_btc', 'usd_price', 'usd_received', 'usd_sent', 'usd_fee',
        'net_change_usd', 'cost_basis_used', 'realized_gain_loss'
    ]

    # Add xpub-specific fields if multi-address mode
    is_xpub = summary.get('xpub') is not None
    if is_xpub:
        fieldnames = base_fields + ['source_address', 'derivation_path']
    else:
        fieldnames = base_fields

    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for tx in summary['transactions']:
            row = {k: tx.get(k, '') for k in base_fields}

            if is_xpub:
                row['source_address'] = tx.get('source_address', '')

                # Get derivation path from address info
                source_addr = tx.get('source_address', '')
                addr_info = summary.get('address_info', {}).get(source_addr, {})
                row['derivation_path'] = addr_info.get('path', 'unknown')

            writer.writerow(row)

    print(f"\n✅ CSV exported to {filename}")
    print("Ready for import into QuickBooks, Xero, or Excel")

    # Also export address list if xpub mode
    if is_xpub and summary.get('discovered_addresses'):
        addr_filename = filename.replace('.csv', '_addresses.csv')
        with open(addr_filename, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['address', 'type', 'index', 'derivation_path'])

            for addr in summary['discovered_addresses']:
                info = summary.get('address_info', {}).get(addr, {})
                writer.writerow([
                    addr,
                    info.get('type', 'unknown'),
                    info.get('index', 'N/A'),
                    info.get('path', 'unknown')
                ])

        print(f"✅ Address list exported to {addr_filename}")


def main():
    print("=" * 70)
    print("BITCOIN WALLET ACCOUNTING EXTRACTOR WITH XPUB SUPPORT")
    print("Fetches transactions + historical prices for financial statements")
    print("=" * 70)
    print()
    print("📍 TWO MODES:")
    print("   1. Single Address: Paste any Bitcoin address")
    print("   2. xpub (HD Wallet): Scan ALL addresses in your wallet")
    print("=" * 70)

    extractor = BitcoinWalletExtractor()

    # Choose mode
    print("\nChoose input type:")
    print("  [1] Single Bitcoin address")
    print("  [2] xpub (scans entire wallet)")

    # Default to demo mode - Satoshi's address
    use_xpub = False
    test_input = "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa"

    print(f"\n🧪 Demo mode: Using Satoshi's Genesis address")
    print(f"   Address: {test_input}")
    print("   Edit the script to use your own address or xpub\n")

    # Check if input is xpub
    if extractor.is_xpub(test_input):
        print("⚡ Detected xpub - scanning all derived addresses...")

        if not BIP32_AVAILABLE:
            print("\n❌ ERROR: xpub scanning requires 'bip32' library")
            print("   Install with: pip install bip32 ecdsa base58")
            print("   For now, using single-address mode...")
            summary = extractor.get_wallet_summary(test_input)
        else:
            print("🔍 Scanning for active addresses (gap limit: 20)...")
            summary = extractor.scan_xpub(test_input, gap_limit=20)
            use_xpub = True
    else:
        print("🔍 Analyzing single address...")
        summary = extractor.get_wallet_summary(test_input)
        use_xpub = False

    print_summary(summary, use_xpub)

    if summary:
        # Export to JSON (full data)
        json_filename = 'wallet_data_xpub.json' if use_xpub else 'wallet_data.json'
        with open(json_filename, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        print(f"\n✅ JSON exported to {json_filename}")

        # Export to CSV (accounting-friendly)
        csv_filename = 'wallet_report_xpub.csv' if use_xpub else 'wallet_report.csv'
        export_to_csv(summary, csv_filename)

        print("\n📝 NEXT STEPS:")
        print(f"1. Open {csv_filename} in Excel/Google Sheets")
        print("2. Review realized gains/losses for tax reporting")
        print("3. Import into QuickBooks, Xero, or your accounting software")
        if use_xpub:
            print("4. Addresses are sorted by derivation path for audit trail")

        print("\n💡 PRODUCTION NOTES:")
        print("   - For real wallets, use your actual xpub from wallet software")
        print("   - Electrum: Wallet → Information → Master Public Key")
        print("   - Ledger/Trezor: Settings → Export public keys")
        print("   - Ledger Live: Accounts → Edit → Extended public key")


if __name__ == "__main__":
    main()


🔔 Note: bip32 library not installed. xpub scanning will be limited.
   Install with: pip install bip32 ecdsa
BITCOIN WALLET ACCOUNTING EXTRACTOR WITH XPUB SUPPORT
Fetches transactions + historical prices for financial statements

📍 TWO MODES:
   1. Single Address: Paste any Bitcoin address
   2. xpub (HD Wallet): Scan ALL addresses in your wallet

Choose input type:
  [1] Single Bitcoin address
  [2] xpub (scans entire wallet)

🧪 Demo mode: Using Satoshi's Genesis address
   Address: 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa
   Edit the script to use your own address or xpub

🔍 Analyzing single address...
Fetching data for address: 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa
Found 50 transactions
Fetching historical prices for cost basis calculations...
Error fetching UTXOs: 400 Client Error: Bad Request for url: https://mempool.space/api/address/1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa/utxo

WALLET SUMMARY: 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa

📊 BITCOIN BALANCES
----------------------------------------
Curr